In [1]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../')

In [2]:
import torch

import pandas as pd

In [3]:
data = pd.read_csv('data/tweets.csv')
data.rename(columns={'tweet_text': 'text'}, inplace=True)
data.drop_duplicates(subset=['text'], inplace=True)
data = data.reset_index(drop=True)
label = 'cyberbullying_type'
text = 'text'
text_prepared = 'text_prepared'

In [102]:
import importlib, sys
importlib.reload(sys.modules['src.dataset'])
from src.dataset import Dataset

In [128]:
ds = Dataset()
ds.texts = ds.texts[:int(len(ds.texts) * 0.4)]
ds.build_pairs()
# a.texts = ["annie, worried, blondes, late, dummy, mkr, sheesh".split(",")]

In [129]:
from torch.utils.data import DataLoader
dl = DataLoader(ds, batch_size=128)

In [130]:
importlib.reload(sys.modules['src.model'])
from src.model import Word2Vec

In [131]:
w2i, i2w = ds.get_w2i_i2w()
w2v = Word2Vec(word_size=len(w2i), hidden_size=1024, w2i=w2i, i2w=i2w)

In [132]:
from torch.optim import AdamW
import torch.nn as nn
import tqdm

optimizer = AdamW(w2v.parameters())
criteria = nn.CrossEntropyLoss()

n_epoch = 10

losses = []
for epoch in range(n_epoch):
    cur_loss = 0
    for x, y in tqdm.tqdm(dl):
        import torch.nn.functional as F
        y = torch.tensor([w2i[i] for i in y])
        y = F.one_hot(y, num_classes=len(w2i)).float()
        optimizer.zero_grad()
        preds = w2v.forward(x)
        loss = criteria(preds, y)
        loss.backward()
        optimizer.step()
        cur_loss += loss.detach()
    
    print(f"loss: {cur_loss / len(dl)}")
        

  0%|                                                 | 0/2500 [00:00<?, ?it/s]/Users/eugene/__important/projects/itmo/2026_sem3/text_processing/.venv_text_processing/lib/python3.13/site-packages/torch/nn/modules/module.py:1787: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
100%|██████████████████████████████████████| 2500/2500 [02:09<00:00, 19.31it/s]


loss: 9.72245979309082


 24%|█████████▏                             | 589/2500 [00:37<02:02, 15.66it/s]


KeyboardInterrupt: 

In [ ]:
ds.pairs['kwaito']